# Exercise 06a — Test and interpret BART station clusters

This exercise uses the same BART catchment-area data as Tutorial 06a. The tutorial fixed $k=3$ to introduce K-means and PCA; here you will test the number of clusters, interpret representative and unusual stations, and examine how scaling and feature choice affect the result.

## Learning objectives

1. use inertia and silhouette score to compare several values of $k$;
2. describe clusters in the original population and employment units;
3. identify representative and unusual stations using distance to a cluster centre;
4. explain why scaling and feature choice can change K-means assignments.

## Open this notebook

- **Colab.** Run the setup cell before any other cell. It clones the public repository and changes into `06 Unsupervised learning`.
- **On your own computer.** Open the notebook from its lesson folder.

In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "06 Unsupervised learning"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())

### Check the packages

The cell should print version numbers and `ready`. The BART station table is the same local `data_X.csv` file used by Tutorial 06a.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

missing_requirements = [
    requirement
    for module, requirement in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_requirements:
    print("Installing:", missing_requirements)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_requirements,
    ])
else:
    print("Required packages are already installed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("ready")

## Prepared data

Each row is one BART station catchment. As in Tutorial 06a, there is no target $y$. The first three questions use population and employment; the extension uses all four available variables.

### What is standardization?

K-means compares stations using Euclidean distance. A variable with a larger numerical spread can therefore contribute more to the distance, even when it is not more important. Standardization puts each feature on a comparable scale. For feature $j$, transform every value using

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j},
$$

where $x_{ij}$ is station $i$'s original value, $\mu_j$ is that feature's mean, and $\sigma_j$ is its standard deviation. After the transformation, each feature has mean approximately 0 and standard deviation approximately 1. A value $z=1.5$ means 1.5 standard deviations above the mean; it does **not** mean 1.5 people or jobs.

In code, `fit` learns the mean and standard deviation, while `transform` applies the formula:

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

This exercise describes one fixed set of 48 stations, so the scaler is fitted to that table. In a prediction task with train/validation/test sets, fit the scaler on the **training set only**, then use the same fitted scaler to transform validation and test data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
pd.set_option("display.max_columns", None)
COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

data = pd.read_csv("data_X.csv").iloc[:48].copy()
cluster_features = ["TotPop", "TotEmp"]
all_features = ["TotPop", "AutoOwn0", "TotEmp", "Tot Rd Dens"]

cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(data[cluster_features])

print("Station table shape:", data.shape, "= (stations, columns)")
print("Two-feature K-means input shape:", X_cluster.shape, "= (stations, standardized features)")
display(data[["Station"] + all_features].head())
display(data[all_features].describe().T.round(1))

scaling_check = pd.DataFrame({
    "raw mean": data[cluster_features].mean(),
    "raw standard deviation": data[cluster_features].to_numpy().std(axis=0),
    "standardized mean": X_cluster.mean(axis=0),
    "standardized standard deviation": X_cluster.std(axis=0),
}, index=cluster_features)
print("Check what StandardScaler changed:")
display(scaling_check.round(3))

## Question 1 — How many station types should we use?

The tutorial fixed $k=3$. Here, fit K-means for $k=2,3,4,5,6$ using the standardized population and employment variables in `X_cluster`. For each $k$, record:

- **inertia**: the total squared distance from stations to their assigned centre (smaller always looks better as $k$ increases);
- **silhouette score**: a balance between compact clusters and separation from other clusters (larger is better).

### What is a silhouette score?

For each station $i$:

1. $a(i)$ is its average distance to the other stations in its **own** cluster. A small $a(i)$ means the station fits its cluster closely.
2. $b(i)$ is the smallest average distance from that station to any **other** cluster. A large $b(i)$ means the nearest competing cluster is far away.
3. The station's silhouette is

$$
s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.
$$

The value lies between $-1$ and $1$: near $1$ means well matched to its own cluster, near $0$ means near a boundary, and below $0$ means it may be closer to another cluster. `silhouette_score(X, labels)` returns the average $s(i)$ across all stations. Compare candidate values of $k$ using the **same feature matrix and distance scale**. A higher score is useful evidence, but it is not an automatic final answer: interpretability and the engineering purpose of the groups also matter. Silhouette is not defined for $k=1$.

Plot both metrics against $k$. Select and save `chosen_k`, then explain whether the numerically best value and the tutorial's $k=3$ tell exactly the same story.

In [ ]:
# TODO: fit K-means for k = 2, 3, 4, 5 and 6.
# TODO: display inertia and silhouette score in a table and two line plots.
# TODO: save your selected number of clusters as chosen_k and justify it.

## Question 2 — What does each station type mean?

Refit K-means with `chosen_k`. Because cluster numbers are arbitrary, reorder the clusters from lower to higher combined population and employment. Then:

1. report each cluster's station count and mean values in the **original units**;
2. find the station nearest to each centre (a representative example);
3. find the station farthest from each centre (an unusual member);
4. plot population against employment, mark the centres, and label those representative and unusual stations.

Use the table and plot to give each cluster a short descriptive name.

In [ ]:
# TODO: refit K-means and reorder cluster IDs from lower to higher activity.
# TODO: report raw-unit profiles and the nearest/farthest station in each cluster.
# TODO: draw the labelled population-versus-employment scatter plot.

## Question 3 — When does scaling change the result?

With this dataset, the selected $k=2$ happens to give the same assignments with or without standardization, so it does not visibly demonstrate the effect of scaling. Return to the tutorial's finer choice, $k=3$, and fit it twice: once with standardized population and employment, and once with their raw values. Reorder both solutions from lower to higher activity and compare them using:

- two side-by-side scatter plots;
- cluster sizes and silhouette scores calculated in the same standardized feature space;
- a table of stations whose ordered type changes.

Explain which variable has the larger numerical scale and how that changes the distances used by K-means. Do not compare the literal cluster numbers before reordering them: labels such as 1 and 2 have no fixed meaning.

In [ ]:
# TODO: fit k = 3 once with standardized and once with raw features.
# TODO: compare plots, sizes and silhouettes, and label stations that change type.
# TODO: explain why scale changes these assignments even though it did not change the k = 2 solution.

## Question 4 — Extension: do two extra features change the station types?

Repeat the clustering with all four standardized variables: population, zero-car households, employment and road density. Keep the same `chosen_k` so the comparison focuses on feature choice.

1. report the four-feature cluster profiles in original units;
2. use PCA only to draw a two-dimensional view of the four-feature data;
3. list the stations whose ordered group changes relative to the two-feature solution;
4. use zero-car households and road density to explain at least two changes.

The PCA coordinates are only for visualization; K-means should still be fitted to all four standardized variables.

In [ ]:
# TODO: standardize all four features and fit K-means with chosen_k.
# TODO: show raw-unit profiles and a two-dimensional PCA visualization.
# TODO: list and interpret stations whose ordered group changes.

## Submission checklist

- an inertia/silhouette table and two $k$-selection plots;
- raw-unit station profiles plus representative and unusual stations;
- a clear standardized-versus-raw comparison;
- a four-feature profile, PCA view and explanation of changed stations;
- short interpretations tied to the displayed results.